In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

# connecties met sdm en dwh databases (brondatabase + dwh database afgeleid van ETL-schema's)

# bron databases
sdm_conn = sqlite3.connect("BikeToDriveDatabase.db")

# data warehouse
dwh_conn = sqlite3.connect("DWH_DB.db")

In [4]:
# als er iets mis is met de connection of foutjes in database run dit zodat de connection wordt gestopt en je opnieuw kan proberen !

dwh_conn.close()

In [ ]:
# full reload: pas aan naar juiste tabel om alle data in te laden in dwh van sdm. 
sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Fiets_Verkoop_Klant
""", sdm_conn)

for index, row in sdm_klant.iterrows():
    dwh_conn.execute("""
    INSERT INTO Klant (
        klantnr, naam, woonplaats, adres, geslacht, geboortedatum
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row['klantnr'],
        row['naam'],
        row['woonplaats'],
        row['adres'],
        row['geslacht'],
        row['geboortedatum']
    ))

dwh_conn.commit()

In [ ]:
# Extract: SDM-data ophalen
sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Fiets_Verkoop_Klant
""", sdm_conn)

# DWH-data ophalen
dwh_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum
FROM Klant
""", dwh_conn)

# merge de twee tabellen
merged = sdm_klant.merge(dwh_klant, on="klantnr", how="left", suffixes=('_sdm', '_dwh'))

# pak alle rijen die nog niet in het DWH staan (nieuwe klanten)
new_rows = merged[merged['naam_dwh'].isna()]

for _, row in new_rows.iterrows():
    dwh_conn.execute("""
    INSERT INTO Klant (klantnr, naam, woonplaats, adres, geslacht, geboortedatum)
    VALUES (?, ?, ?, ?, ?, ?)
    """, (
        row['klantnr'],
        row['naam_sdm'],
        row['woonplaats_sdm'],
        row['adres_sdm'],
        row['geslacht_sdm'],
        row['geboortedatum_sdm']
    ))

# scd_type1
changed_rows = merged[
    (merged['naam_dwh'].notna()) &
    (
        (merged['naam_sdm'] != merged['naam_dwh']) |
        (merged['woonplaats_sdm'] != merged['woonplaats_dwh']) |
        (merged['adres_sdm'] != merged['adres_dwh']) |
        (merged['geslacht_sdm'] != merged['geslacht_dwh']) |
        (merged['geboortedatum_sdm'] != merged['geboortedatum_dwh'])
    )
]

for _, row in changed_rows.iterrows():
    dwh_conn.execute("""
    UPDATE Klant
    SET naam = ?, woonplaats = ?, adres = ?, geslacht = ?, geboortedatum = ?
    WHERE klantnr = ?
    """, (
        row['naam_sdm'],
        row['woonplaats_sdm'],
        row['adres_sdm'],
        row['geslacht_sdm'],
        row['geboortedatum_sdm'],
        row['klantnr']
    ))

dwh_conn.commit()

In [7]:
for index, row in changed_rows.iterrows():
    dwh_conn.execute("""
    UPDATE Klant
    SET naam = ?, woonplaats = ?, adres = ?, geslacht = ?, geboortedatum = ?
    WHERE klantnr = ?
    """, (
        row['naam_sdm'],
        row['woonplaats_sdm'],
        row['adres_sdm'],
        row['geslacht_sdm'],
        row['geboortedatum_sdm'],
        row['klantnr']
    ))

dwh_conn.commit()